# 02 Model Architecture: DynaMo & PMPGen

This notebook visualizes the architectures of both models:
- DynaMo Phase 1: binding residue prediction
- PMPGen Phase 2: de novo protein generation
- Component breakdown and information flow

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

## 1. DynaMo Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')

# Define colors
color_input = '#E8F4F8'
color_process = '#B3E5FC'
color_novel = '#FFE082'  # novel components in yellow
color_output = '#C8E6C9'

# Title
ax.text(5, 11.5, 'DynaMo Phase 1: Membrane Binding Prediction', 
        fontsize=16, fontweight='bold', ha='center')

# Input layer
boxes = []
inputs = ['PLM\n(1280)', 'Struct\n(19)', 'Vectors\n(6×3)']
for i, inp in enumerate(inputs):
    x = 1 + i * 2.5
    box = FancyBboxPatch((x-0.6, 10), 1.2, 0.7, boxstyle='round,pad=0.05', 
                          edgecolor='black', facecolor=color_input, linewidth=2)
    ax.add_patch(box)
    ax.text(x, 10.35, inp, ha='center', va='center', fontsize=9, fontweight='bold')

# Feature fusion
box = FancyBboxPatch((2.5-1, 8.8), 2, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 9.15, 'Feature Fusion\n(Independent LayerNorm)', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# GVP Encoder
box = FancyBboxPatch((2.5-1, 7.8), 2, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 8.15, 'GVP-GNN Encoder\n(3 layers, 256-dim)', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Two paths: Dynamics and Geometry
# Left path: Dynamics
box = FancyBboxPatch((0.5-0.8, 6.5), 1.6, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_novel, linewidth=2)
ax.add_patch(box)
ax.text(1.3, 6.85, 'Conf. Attention\nPool ✨', 
        ha='center', va='center', fontsize=8, fontweight='bold')
ax.text(1.3, 6.2, 'RMSF-adaptive\ntemp.', ha='center', fontsize=7, style='italic')

# Right path: Geometry
box = FancyBboxPatch((5.5-0.8, 6.5), 1.6, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_novel, linewidth=2)
ax.add_patch(box)
ax.text(6.3, 6.85, 'Membrane\nGeometry ✨', 
        ha='center', va='center', fontsize=8, fontweight='bold')
ax.text(6.3, 6.2, '(depth, tilt,\namph)', ha='center', fontsize=7, style='italic')

# Arrows from GVP to both paths
arrow1 = FancyArrowPatch((2.8, 7.8), (1.8, 7.2), 
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow1)
arrow2 = FancyArrowPatch((4.2, 7.8), (6.2, 7.2),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow2)

# Cross-attention
box = FancyBboxPatch((2.5-1, 4.8), 2, 0.8, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_novel, linewidth=3)
ax.add_patch(box)
ax.text(3.5, 5.25, 'Cross-Attention\nFusion ✨', 
        ha='center', va='center', fontsize=9, fontweight='bold')
ax.text(3.5, 4.9, 'Structure→Dynamics', ha='center', fontsize=7, style='italic')

# Arrows to cross-attention
arrow3 = FancyArrowPatch((1.3, 6.5), (2.8, 5.6),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow3)
arrow4 = FancyArrowPatch((6.3, 6.5), (4.2, 5.6),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow4)

# Physicochemical gate
box = FancyBboxPatch((2.5-1, 3.8), 2, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_novel, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 4.15, 'Phys. Gate\n(KD, charge, SASA)', 
        ha='center', va='center', fontsize=8, fontweight='bold')

# Arrow
arrow5 = FancyArrowPatch((3.5, 4.8), (3.5, 4.5),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow5)

# Classifier
box = FancyBboxPatch((2.5-1, 2.8), 2, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 3.15, 'MLP Classifier\n(256→64→1)', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Arrow
arrow6 = FancyArrowPatch((3.5, 3.8), (3.5, 3.5),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow6)

# Output
box = FancyBboxPatch((2.5-0.8, 1.8), 1.6, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_output, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 2.15, 'Binding\nProbability', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Arrow
arrow7 = FancyArrowPatch((3.5, 2.8), (3.5, 2.5),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow7)

# Legend
ax.text(0.5, 0.8, '✨ = Novel Component', fontsize=10, fontweight='bold', color='darkgoldenrod')
ax.text(0.5, 0.3, 'Losses: Focal + Patch Contiguity + Contrastive', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig('outputs/dynamo_architecture.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ DynaMo architecture diagram saved')

## 2. PMPGen Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')

# Title
ax.text(5, 11.5, 'PMPGen Phase 2: De Novo Protein Generation via SE(3) Flow Matching', 
        fontsize=14, fontweight='bold', ha='center')

# Conditioning inputs
inputs = ['Query\nScaffold', 'Geometry\n(OPM)', 'Binding\nPatch', 'RMSF']
for i, inp in enumerate(inputs):
    x = 0.8 + i * 2.2
    box = FancyBboxPatch((x-0.5, 10), 1, 0.7, boxstyle='round,pad=0.05',
                          edgecolor='black', facecolor=color_input, linewidth=2)
    ax.add_patch(box)
    ax.text(x, 10.35, inp, ha='center', va='center', fontsize=8, fontweight='bold')

# Conditioning encoders
box = FancyBboxPatch((1.5-1, 8.8), 2, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(2.5, 9.15, 'Cond. Encoders\n& Fusion', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Noise schedule
box = FancyBboxPatch((5.5-0.8, 8.8), 1.6, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_novel, linewidth=2)
ax.add_patch(box)
ax.text(6.3, 9.15, 'MD Noise\nSchedule ✨', 
        ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows down
arrow1 = FancyArrowPatch((2.5, 8.8), (2.5, 8.1),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow1)
arrow2 = FancyArrowPatch((6.3, 8.8), (6.3, 8.1),
                         arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow2)

# Flow matching loop (left side)
box = FancyBboxPatch((0.5-0.8, 7), 1.6, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(1.3, 7.35, 'OT Flow\nInterpolant', ha='center', va='center', fontsize=8, fontweight='bold')

# IPA Denoiser (center)
box = FancyBboxPatch((2.5-1, 7), 2, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 7.35, 'IPA Denoiser\n(6 layers)', ha='center', va='center', fontsize=9, fontweight='bold')

# Membrane guidance (right side)
box = FancyBboxPatch((5.5-0.8, 7), 1.6, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_novel, linewidth=2)
ax.add_patch(box)
ax.text(6.3, 7.35, 'Mem.\nGuidance ✨', ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows to flow
arrow3 = FancyArrowPatch((2.5, 8.1), (1.8, 7.7), arrowstyle='->', mutation_scale=20, linewidth=1.5)
ax.add_patch(arrow3)
arrow4 = FancyArrowPatch((6.3, 8.1), (4.2, 7.7), arrowstyle='->', mutation_scale=20, linewidth=1.5)
ax.add_patch(arrow4)

# Denoising loop
ax.text(4.5, 6.2, 'Iterate T=500 steps', ha='center', fontsize=9, style='italic', fontweight='bold')
arc = mpatches.Arc((3.5, 5.8), 3, 2, angle=0, theta1=45, theta2=315, linewidth=2.5, color='darkblue')
ax.add_patch(arc)

# Velocity prediction outputs
outputs = ['v_R', 'v_t']
for i, out in enumerate(outputs):
    x = 2 + i * 1.5
    box = FancyBboxPatch((x-0.4, 5), 0.8, 0.5, boxstyle='round,pad=0.05',
                          edgecolor='black', facecolor=color_process, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, 5.25, out, ha='center', va='center', fontsize=8, fontweight='bold')

# Backbone frames
box = FancyBboxPatch((2.5-1, 3.8), 2, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 4.15, 'Generate Backbone\nFrames (R, t)', 
        ha='center', va='center', fontsize=9, fontweight='bold')

arrow5 = FancyArrowPatch((3.5, 5), (3.5, 4.5), arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow5)

# Sequence decoder
box = FancyBboxPatch((2.5-0.9, 2.8), 1.8, 0.7, boxstyle='round,pad=0.05',
                      edgecolor='black', facecolor=color_process, linewidth=2)
ax.add_patch(box)
ax.text(3.5, 3.15, 'ProteinMPNN\nSequence Decoder', 
        ha='center', va='center', fontsize=8, fontweight='bold')

arrow6 = FancyArrowPatch((3.5, 3.8), (3.5, 3.5), arrowstyle='->', mutation_scale=20, linewidth=2)
ax.add_patch(arrow6)

# Validation
ax.text(3.5, 2.3, '3-Stage Validation Cascade ✨', ha='center', fontsize=9, 
        fontweight='bold', color='darkred', bbox=dict(boxstyle='round', facecolor='lightyellow'))

validations = ['ESMFold\npLDDT>70', 'DynaMo\nRecall>0.8', 'Rosetta ΔG\nDiversity']
for i, val in enumerate(validations):
    x = 0.8 + i * 3
    box = FancyBboxPatch((x-0.5, 1.1), 1, 0.7, boxstyle='round,pad=0.05',
                          edgecolor='black', facecolor='#E8D5C4', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, 1.45, val, ha='center', va='center', fontsize=7, fontweight='bold')

# Legend
ax.text(0.5, 0.4, '✨ = Novel Component', fontsize=10, fontweight='bold', color='darkgoldenrod')
ax.text(0.5, -0.1, 'Losses: Flow + Anchor + Membrane + Structure', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig('outputs/pmpgen_architecture.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ PMPGen architecture diagram saved')

## 3. Feature Dimensions Summary

In [ ]:
# Summary of feature dimensions
fig, ax = plt.subplots(figsize=(12, 8))

# Data
components = [
    'PLM (ESM-2)\n→ 128-dim',
    'Backbone\nDihedrals (6)',
    'Physico-\nchemical (4)',
    'Solvent\nExposure (5)',
    'PMP-Specific\n(4)',
    'Total Scalar\nFeatures',
    '\n',
    'Backbone\nFrame (3×3)',
    'Virtual Cβ',
    'Membrane\nNormal',
    'MD Displacement',
    'Total Vector\nFeatures'
]

dims = [128, 6, 4, 5, 4, 147, 0, 3, 3, 3, 3, 6]
colors_list = ['#42A5F5']*5 + ['#2196F3', 'white'] + ['#81C784']*4 + ['#4CAF50']

y_pos = np.arange(len(components))
bars = ax.barh(y_pos, [d if d > 0 else 1 for d in dims], color=colors_list, edgecolor='black', linewidth=1.5)

# Add value labels
for i, (bar, dim) in enumerate(zip(bars, dims)):
    if dim > 0:
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2, 
               f'{dim}', va='center', fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(components, fontsize=10)
ax.set_xlabel('Dimensions', fontsize=12, fontweight='bold')
ax.set_title('DynaMo Input Feature Dimensions', fontsize=14, fontweight='bold')
ax.set_xlim(0, 160)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/feature_dimensions.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Feature dimensions summary saved')

## 4. Parameter Counts

In [ ]:
# Model parameter comparison
model_params = {
    'DynaMo Full': 2_800_000,
    '  - GVP Encoder': 1_200_000,
    '  - Conf. Attention': 380_000,
    '  - Geometry Path': 280_000,
    '  - Cross-Attention': 200_000,
    '  - Classifier': 25_000,
    '  - Others': 335_000,
    '\n': 0,
    'PMPGen Full': 4_500_000,
    '  - IPA Denoiser': 2_500_000,
    '  - Conditioning': 800_000,
    '  - Noise Schedule': 150_000,
    '  - Others': 1_050_000,
}

fig, ax = plt.subplots(figsize=(10, 8))

names = []
params = []
indent_level = []

for name, param in model_params.items():
    if param > 0:
        names.append(name)
        params.append(param / 1_000_000)  # convert to millions
        indent = name.count('  ')
        indent_level.append(indent)

colors = ['#FF6F00' if 'DynaMo' in n else '#1976D2' for n in names]
colors = ['#FF6F00' if 'Full' in n else '#FFB74D' if 'DynaMo' in n else '#1976D2' if 'Full' in n else '#64B5F6' for n in names]

y_pos = np.arange(len(names))
bars = ax.barh(y_pos, params, color=colors, edgecolor='black', linewidth=1.5)

for bar, param in zip(bars, params):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
           f'{param:.1f}M', va='center', fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Parameters (Millions)', fontsize=12, fontweight='bold')
ax.set_title('Model Parameter Counts', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/model_parameters.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'DynaMo total: 2.8M parameters')
print(f'PMPGen total: 4.5M parameters')
print(f'Combined: 7.3M parameters')